In [1]:
! python --version

Python 3.12.8


In [14]:
from __future__ import annotations

"""Utility helpers for the recipe chatbot backend.

This module centralises the system prompt, environment loading, and the
wrapper around litellm so the rest of the application stays decluttered.
"""

import os
from typing import Final, List, Dict

import litellm  # type: ignore
from dotenv import load_dotenv

# Ensure the .env file is loaded as early as possible.
load_dotenv(override=False)

# --- Constants -------------------------------------------------------------------

meal_part_options = ['entrée', 'dessert', 'beverage']
dietary_preference_options = ['vegan', 'vegetarian', 'omnivore', 'pescatarian']
style_options = ['quick_and_easy', 'gourmet', 'comfort_food']

SYSTEM_PROMPT: Final[str] = f'''\
Given the following key Japanese recipe dimensions:
- Style: one of {style_options}
- Dietary Preference: one of {dietary_preference_options}
- Meal Part: one of {meal_part_options}

Can you provide a list of exactly 20 combinations of these dimensions? They have to make sense together.
Remember, that we only need 20 combinations.
'''

# Fetch configuration *after* we loaded the .env file.
MODEL_NAME: Final[str] = os.environ.get("MODEL_NAME", "gpt-4o-mini")

def get_agent_response() -> list[tuple[str, str, str]]:
    """Use the SYSTEM_PROMPT and an LLM call to return a realistic list of 20 unique combinations of the key Japanese recipe dimensions (style, dietary_preference, meal_part) as a list of tuples."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Please provide the list as a Python list of tuples, where each tuple is (style, dietary_preference, meal_part)."}
    ]
    completion = litellm.completion(
        model=MODEL_NAME,
        messages=messages,
    )
    import ast
    import re
    # Extract the list of tuples from the assistant's reply
    reply = completion["choices"][0]["message"]["content"].strip()
    # Try to extract the first Python list of tuples from the reply
    match = re.search(r'\[.*\]', reply, re.DOTALL)
    if match:
        list_str = match.group(0)
        try:
            result = ast.literal_eval(list_str)
            if isinstance(result, list) and all(isinstance(t, tuple) and len(t) == 3 for t in result):
                return result
        except Exception:
            pass
    # Fallback: return the raw reply if parsing fails
    return reply

length = len(get_agent_response()) if isinstance(get_agent_response(), list) else 'N/A'
print(f"Number of combinations: {length}")
get_agent_response()

Number of combinations: 20


[('quick_and_easy', 'vegan', 'entrée'),
 ('quick_and_easy', 'vegetarian', 'entrée'),
 ('quick_and_easy', 'pescatarian', 'entrée'),
 ('quick_and_easy', 'omnivore', 'entrée'),
 ('quick_and_easy', 'vegetarian', 'dessert'),
 ('quick_and_easy', 'vegan', 'dessert'),
 ('quick_and_easy', 'omnivore', 'dessert'),
 ('quick_and_easy', 'pescatarian', 'dessert'),
 ('quick_and_easy', 'pescatarian', 'beverage'),
 ('quick_and_easy', 'vegan', 'beverage'),
 ('gourmet', 'omnivore', 'entrée'),
 ('gourmet', 'pescatarian', 'entrée'),
 ('gourmet', 'vegetarian', 'entrée'),
 ('gourmet', 'omnivore', 'dessert'),
 ('gourmet', 'vegetarian', 'dessert'),
 ('gourmet', 'vegan', 'dessert'),
 ('gourmet', 'omnivore', 'beverage'),
 ('comfort_food', 'omnivore', 'entrée'),
 ('comfort_food', 'vegetarian', 'entrée'),
 ('comfort_food', 'vegan', 'entrée')]

In [15]:
def generate_natural_language_queries(tuples: list[tuple[str, str, str]], n: int = 10) -> list[str]:
    """Use the LLM to generate realistic natural language user queries for the best n tuples."""
    # Select the first n tuples (or allow user to pass in their own selection logic)
    selected_tuples = tuples[:n]
    # Format the tuples for the prompt
    tuple_list_str = '\n'.join([f"{i+1}. {t}" for i, t in enumerate(selected_tuples)])
    prompt = f'''Given the following recipe request dimensions (style, dietary_preference, meal_part) as tuples:\n\n{tuple_list_str}\n\nFor each tuple, write a realistic and natural user query that a human might ask a recipe chatbot. Each query should be specific and reflect the tuple's values, but should not mention the tuple structure or variable names. Return the queries as a Python list of strings, in the same order as the tuples above.'''
    messages = [
        {"role": "system", "content": "You are an expert at generating realistic user queries for a recipe chatbot."},
        {"role": "user", "content": prompt}
    ]
    completion = litellm.completion(
        model=MODEL_NAME,
        messages=messages,
    )
    import ast
    import re
    reply = completion["choices"][0]["message"]["content"].strip()
    # Try to extract the first Python list of strings from the reply
    match = re.search(r'\[.*\]', reply, re.DOTALL)
    if match:
        list_str = match.group(0)
        try:
            result = ast.literal_eval(list_str)
            if isinstance(result, list) and all(isinstance(q, str) for q in result):
                return result
        except Exception:
            pass
    return reply

# Example usage:
tuples = get_agent_response() if isinstance(get_agent_response(), list) else []
queries = generate_natural_language_queries(tuples, n=10)
print("\nGenerated user queries:")
for q in queries:
    print(q)


Generated user queries:
Can you suggest a quick and easy vegan main dish I can make for dinner?
I'm looking for a simple vegan dessert recipe that I can whip up fast. Any ideas?
What’s a quick and easy vegetarian entree I can prepare on a busy weeknight?
Do you have any easy and vegetarian drink recipes that I can make quickly?
I need a fast and easy main course recipe with meat for dinner. What do you recommend?
Could you give me a quick and easy meat-based beverage recipe, maybe something warm or a cocktail?
What’s a quick and simple pescatarian main dish I can make for lunch?
I'm looking for a fast and easy seafood-based drink recipe. Any suggestions?
Can you provide a gourmet vegetarian main course recipe that’s elegant but doable?
I’d love a fancy vegetarian dessert recipe that looks impressive but isn’t too complicated. What do you have?
